# Week 2 — Use an LLM as an annotator

**Research task:** Apply the same housing-opinion codebook through OpenRouter and Ollama, then distinguish route agreement from agreement with a human reference.

**Python introduced:** lists, dictionaries, indexing, dictionary keys and equality comparisons.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session02/session02_annotation_measurement.ipynb)

Colab supports the OpenRouter route only. Local JupyterLab or VS Code is canonical because it can also reach Ollama.

In [ ]:
# Colab setup: clone the public repository when running in Colab.
import os as setup_os
import subprocess as setup_subprocess
from pathlib import Path as SetupPath
if SetupPath('/content').exists():
    setup_repo = SetupPath('/content/GenAI_Soc2026')
    if not setup_repo.exists():
        setup_subprocess.run(['git','clone','https://github.com/cjbarrie/GenAI_Soc2026.git',str(setup_repo)], check=True)
    setup_os.chdir(setup_repo / 'workbook' / 'session02')
print('Working folder:', SetupPath.cwd())

## Load the course settings and SDKs

**Input:** installed Python packages, `config/course_models.json`, and—if it is not already set—the hidden OpenRouter key. **Operations:** `import` makes an installed tool available; `Path.cwd()` gives Python the current folder; the `while` block walks upward until it finds the course configuration; `json.loads(...)` turns the file's JSON text into a dictionary; square brackets retrieve the two model names. **Output:** `HOSTED_MODEL` and `LOCAL_MODEL` are strings. `getpass(...)` accepts the key without echoing it. The folder-search code is supplied setup and is not assessed.


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## Store one comment, the codebook and a human reference label

`comment` is the text to classify. `codebook` is a dictionary mapping each permitted label to its definition. `human_label` is a separate reference judgment; it is not sent to the model. `codebook["UNCLEAR"]` uses a key to retrieve one definition and demonstrates how dictionaries differ from lists.


In [ ]:
comment = "I support the plan if rents remain affordable."
codebook = {
    "SUPPORT": "unconditional support for the proposal",
    "OPPOSE": "unconditional opposition to the proposal",
    "UNCLEAR": "conditional, mixed, procedural, or insufficient evidence",
}
human_label = "UNCLEAR"
print(comment)
print(codebook["UNCLEAR"])

## Construct the exact message list used by both routes

`json.dumps(codebook)` turns the Python dictionary into readable text that can be placed inside the prompt. The `+` signs join codebook, instruction and comment. The resulting `messages` list contains one user message. `[0]` retrieves the first list item; `["content"]` then retrieves that dictionary's content field.


In [ ]:
prompt = (
    "Apply this codebook: " + json.dumps(codebook) +
    "\nReturn only SUPPORT, OPPOSE, or UNCLEAR.\nComment: " + comment
)
messages = [{"role": "user", "content": prompt}]
print(messages[0]["content"])

## Make and unpack the OpenRouter call

The inputs are the hosted model, the shared message list and zero temperature. `[0]` chooses the first returned choice, `.message.content` retrieves its text, `.strip()` removes spare whitespace and `.upper()` standardizes letter case. The final output, `hosted_label`, remains a string and may still fall outside the codebook.


In [ ]:
with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
    hosted_response = client.chat.send(
        model=HOSTED_MODEL, messages=messages, temperature=0,
    )
hosted_choice = hosted_response.choices[0]
hosted_raw = hosted_choice.message.content
print("OpenRouter raw return:", hosted_raw)
hosted_label = hosted_raw.strip().upper()
print("OpenRouter label:", hosted_label)

## Make and unpack the Ollama call

The same messages enter Ollama. Its temperature sits inside an `options` dictionary and its returned text is found at `local_response.message.content`. Standardizing case makes the two strings easier to compare; it does not repair a substantively wrong label.


In [ ]:
local_response = ollama.chat(
    model=LOCAL_MODEL, messages=messages, options={"temperature": 0},
)
local_raw = local_response.message.content
print("Ollama raw return:", local_raw)
local_label = local_raw.strip().upper()
print("Ollama label:", local_label)

## Compare route agreement and human-reference agreement

Each `==` asks whether two strings are equal and returns `True` or `False`. `routes_agree` compares the two models. The other two values compare each model with the independently supplied human reference. These are three distinct questions, so they are stored and printed separately.


In [ ]:
routes_agree = hosted_label == local_label
hosted_matches_human = hosted_label == human_label
local_matches_human = local_label == human_label
print("Routes agree:", routes_agree)
print("OpenRouter matches human label:", hosted_matches_human)
print("Ollama matches human label:", local_matches_human)

# ONE CHANGE: replace comment with
# "I oppose the rezoning proposal because it will displace tenants."
# Then rerun from the message-construction cell.

## Methodological check

Agreement between the routes is a reproducibility observation. Agreement with one human label is criterion evidence. Neither alone proves that the codebook measures the construct well.
## Completion recording

Run both routes on the changed opposition comment. Explain the comment string, codebook dictionary, message list, `[0]`, each response path and all three Boolean comparisons. State whether disagreement is between routes, with the human reference, or both.

Explain every input and output aloud. Never show the shared key.